# I02 - Regularization Techniques

**Level:** Intermediate

---

## Learning Objectives

By the end of this lesson, you will:
1. Understand overfitting and how to prevent it
2. Implement L1 and L2 regularization
3. Master dropout and its variants
4. Learn data augmentation techniques
5. Apply early stopping effectively

---

## Prerequisites

- Completed B01-B07 (Basic Level)
- Understanding of neural networks
- Familiarity with overfitting concepts

---

In [ ]:
# Import required libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"TensorFlow version: {tf.__version__}")

## 1. Understanding Overfitting

### What is Overfitting?

**Overfitting** occurs when a model learns the training data too well, including noise and outliers, resulting in poor generalization to new data.

**Signs of Overfitting:**
- High training accuracy, low validation accuracy
- Training loss decreases, validation loss increases
- Large gap between training and validation metrics

**Causes:**
- Model too complex for the data
- Too many parameters
- Insufficient training data
- Training for too many epochs

In [ ]:
# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Use smaller subset to demonstrate overfitting
x_train_small = x_train[:5000]
y_train_small = y_train[:5000]

print(f"Training samples: {x_train_small.shape}")
print(f"Test samples: {x_test.shape}")

## 2. L1 and L2 Regularization

### L2 Regularization (Ridge)

Adds penalty proportional to square of weights:

$$L_{total} = L_{original} + \lambda \sum_{i} w_i^2$$

**Effect:** Encourages small weights, prevents any single weight from becoming too large

### L1 Regularization (Lasso)

Adds penalty proportional to absolute value of weights:

$$L_{total} = L_{original} + \lambda \sum_{i} |w_i|$$

**Effect:** Encourages sparsity, some weights become exactly zero

In [ ]:
# Model without regularization (baseline)
def create_baseline_model():
    model = keras.Sequential([
        layers.Flatten(input_shape=(32, 32, 3)),
        layers.Dense(512, activation='relu'),
        layers.Dense(256, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

# Model with L2 regularization
def create_l2_model(l2_lambda=0.01):
    model = keras.Sequential([
        layers.Flatten(input_shape=(32, 32, 3)),
        layers.Dense(512, activation='relu', 
                    kernel_regularizer=regularizers.l2(l2_lambda)),
        layers.Dense(256, activation='relu',
                    kernel_regularizer=regularizers.l2(l2_lambda)),
        layers.Dense(128, activation='relu',
                    kernel_regularizer=regularizers.l2(l2_lambda)),
        layers.Dense(10, activation='softmax')
    ])
    return model

# Model with L1 regularization
def create_l1_model(l1_lambda=0.01):
    model = keras.Sequential([
        layers.Flatten(input_shape=(32, 32, 3)),
        layers.Dense(512, activation='relu',
                    kernel_regularizer=regularizers.l1(l1_lambda)),
        layers.Dense(256, activation='relu',
                    kernel_regularizer=regularizers.l1(l1_lambda)),
        layers.Dense(128, activation='relu',
                    kernel_regularizer=regularizers.l1(l1_lambda)),
        layers.Dense(10, activation='softmax')
    ])
    return model

In [ ]:
# Train and compare models
models = {
    'Baseline (No Regularization)': create_baseline_model(),
    'L2 Regularization': create_l2_model(0.001),
    'L1 Regularization': create_l1_model(0.001)
}

histories = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    history = model.fit(
        x_train_small, y_train_small,
        epochs=50,
        batch_size=128,
        validation_split=0.2,
        verbose=0
    )
    
    histories[name] = history.history
    
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for name, history in histories.items():
    axes[0].plot(history['loss'], label=f'{name} (train)', linewidth=2)
    axes[0].plot(history['val_loss'], label=f'{name} (val)', 
                linestyle='--', linewidth=2)

axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training vs Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for name, history in histories.items():
    axes[1].plot(history['accuracy'], label=f'{name} (train)', linewidth=2)
    axes[1].plot(history['val_accuracy'], label=f'{name} (val)',
                linestyle='--', linewidth=2)

axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training vs Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Dropout Regularization

### How Dropout Works

**Dropout** randomly sets a fraction of input units to 0 during training:

1. During training: Randomly drop neurons with probability `p`
2. During inference: Use all neurons, scale outputs by `(1-p)`

**Benefits:**
- Prevents co-adaptation of neurons
- Acts like ensemble learning
- Very effective regularization

**Typical dropout rates:**
- Hidden layers: 0.2 - 0.5
- Input layer: 0.1 - 0.2

In [ ]:
# Model with dropout
def create_dropout_model(dropout_rate=0.5):
    model = keras.Sequential([
        layers.Flatten(input_shape=(32, 32, 3)),
        layers.Dense(512, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(256, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(128, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(10, activation='softmax')
    ])
    return model

# Train model with dropout
model_dropout = create_dropout_model(0.5)
model_dropout.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_dropout = model_dropout.fit(
    x_train_small, y_train_small,
    epochs=50,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

## 4. Data Augmentation

### Why Data Augmentation?

**Data augmentation** artificially increases training data by applying transformations:

**Common transformations:**
- Rotation
- Flipping (horizontal/vertical)
- Zooming
- Shifting
- Brightness/contrast adjustment

**Benefits:**
- More training data without collecting new samples
- Model learns invariance to transformations
- Reduces overfitting

In [ ]:
# Create data augmentation layer
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1)
])

# Visualize augmented images
plt.figure(figsize=(15, 5))
for i in range(9):
    augmented_image = data_augmentation(x_train[0:1])
    plt.subplot(2, 5, i + 1)
    plt.imshow(augmented_image[0])
    plt.axis('off')
    if i == 0:
        plt.title('Original', fontsize=10)
    else:
        plt.title(f'Augmented {i}', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Model with data augmentation
def create_augmented_model():
    inputs = keras.Input(shape=(32, 32, 3))
    x = data_augmentation(inputs)
    x = layers.Flatten()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    return model

model_augmented = create_augmented_model()
model_augmented.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_augmented = model_augmented.fit(
    x_train_small, y_train_small,
    epochs=50,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

## 5. Early Stopping

### What is Early Stopping?

**Early stopping** monitors validation performance and stops training when it stops improving:

**Parameters:**
- `monitor`: Metric to watch (e.g., 'val_loss')
- `patience`: Number of epochs to wait before stopping
- `restore_best_weights`: Restore weights from best epoch

**Benefits:**
- Prevents overfitting
- Saves training time
- Automatically finds optimal number of epochs

In [ ]:
# Create callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

# Train with callbacks
model_callbacks = create_dropout_model(0.5)
model_callbacks.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_callbacks = model_callbacks.fit(
    x_train_small, y_train_small,
    epochs=100,  # Set high, early stopping will stop earlier
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

## 6. Combining Regularization Techniques

### Best Practices

**Recommended combination:**
1. Start with dropout (0.2-0.5)
2. Add L2 regularization (0.001-0.01)
3. Use data augmentation for images
4. Apply early stopping
5. Use batch normalization (covered in I03)

**Don't overdo it:**
- Too much regularization → underfitting
- Start simple, add regularization if overfitting occurs

In [ ]:
# Complete model with all regularization techniques
def create_complete_model():
    inputs = keras.Input(shape=(32, 32, 3))
    
    # Data augmentation
    x = data_augmentation(inputs)
    x = layers.Flatten()(x)
    
    # Dense layers with L2 and dropout
    x = layers.Dense(512, activation='relu',
                    kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.Dropout(0.5)(x)
    
    x = layers.Dense(256, activation='relu',
                    kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.Dropout(0.5)(x)
    
    x = layers.Dense(128, activation='relu',
                    kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.Dropout(0.3)(x)
    
    outputs = layers.Dense(10, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    return model

print("Complete regularized model created successfully!")

## Summary

In this lesson, you learned:

1. ✅ Understanding and identifying overfitting
2. ✅ L1 and L2 regularization techniques
3. ✅ Dropout and its effectiveness
4. ✅ Data augmentation for images
5. ✅ Early stopping and learning rate reduction
6. ✅ Combining multiple regularization techniques

**Key Takeaways:**
- Regularization prevents overfitting
- Dropout is highly effective and easy to use
- Data augmentation increases effective dataset size
- Combine techniques for best results
- Monitor validation metrics closely

**Next Steps:**
- Move to I03 - Batch and Layer Normalization
- Experiment with different regularization strengths
- Try regularization on your own datasets

---

**References:**
- Srivastava et al. (2014): "Dropout: A Simple Way to Prevent Neural Networks from Overfitting"
- Goodfellow et al. (2016): "Deep Learning" - Chapter 7
- TensorFlow Regularization Guide

---

## Key Takeaways

**Relevant UoA Courses:** COMPSCI 762, COMPSCI 713

1. L1 regularization encourages sparsity: many weights become exactly zero
2. L2 regularization (weight decay) prevents large weights: w = w(1-λ)
3. Dropout randomly drops neurons: prevents co-adaptation
4. Data augmentation: rotation, flip, crop, color jitter for images
5. Early stopping: monitor validation loss, stop when it stops improving

---

## Exam Preparation Guide

### Essential Concepts for Exams

- Know L1 vs L2: L1→sparse, L2→small weights
- Calculate regularized loss: Loss + λ·Σw² (L2) or Loss + λ·Σ|w| (L1)
- Understand dropout: p=0.5 means 50% neurons dropped during training
- Explain why data augmentation helps: increases effective dataset size
- Common question: Given overfitting, which regularization techniques to apply?

### Common Mistakes to Avoid

- ❌ Using dropout during inference (should be disabled)
- ❌ Applying data augmentation to validation/test sets
- ❌ Setting regularization strength too high (underfitting)

### Practice Problems

1. Model overfits: train acc 99%, val acc 75%. Suggest 3 solutions
2. Calculate L2 loss: MSE=5, weights=[1,2,3], λ=0.01
3. Why does dropout prevent overfitting?

### How This Helps Your UoA Courses

**COMPSCI 762, COMPSCI 713:**
- Provides hands-on implementation of theoretical concepts
- Practice problems similar to exam questions
- Reinforces lecture material with code examples
- Helps build intuition for complex topics

### Study Tips for Advanced Topics

1. **Connect to Fundamentals**: Link advanced concepts to basics
2. **Read Papers**: Understand state-of-the-art approaches
3. **Implement from Scratch**: Deepens understanding
4. **Compare Approaches**: Know trade-offs between methods
5. **Real-World Applications**: Understand practical use cases

### Exam Question Types

- **Conceptual**: Explain advanced mechanisms and why they work
- **Comparison**: Compare multiple approaches, trade-offs
- **Design**: Design system for specific requirements
- **Analysis**: Analyze experimental results, identify issues
- **Application**: Apply techniques to novel problems

---


---

## Learning Progress Tracker

Use this section to track your learning progress for this lesson.

### Completion Status
- [ ] Lesson completed
- [ ] All code cells executed successfully
- [ ] Understood all key concepts
- [ ] Completed practice exercises (if any)

### Dates
- **First Completed:** ____/____/____
- **Last Reviewed:** ____/____/____
- **Next Review:** ____/____/____ (Recommended: 1 week, 1 month, 3 months)

### Understanding Level
Rate your understanding (1-5): _____ / 5

- 1 = Need to review completely
- 2 = Understood basics, need more practice
- 3 = Good understanding, minor gaps
- 4 = Strong understanding, can explain to others
- 5 = Mastered, can apply in projects

### Notes & Reflections
```
Write your notes here:
- What concepts were challenging?
- What was interesting or surprising?
- How can you apply this in projects?
- Questions to explore further?




```

### Key Concepts to Remember (I02)
- [ ] L1/L2 regularization
- [ ] Dropout variants
- [ ] Data augmentation
- [ ] Early stopping

---